<a href="https://colab.research.google.com/github/KARTIKPANSURIYA/lcldd/blob/main/lcldd_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

Tue Mar 31 13:28:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install transformers accelerate huggingface_hub sentencepiece -q

In [ ]:
# Test if repo is accessible
!git clone https://github.com/KARTIKPANSURIYA/lcldd.git /content/lcldd

import os
if os.path.exists("/content/lcldd"):
    print("\n✅ Repo cloned successfully!")
    print("Files found:")
    for f in os.listdir("/content/lcldd"):
        print(f"  {f}")
else:
    print("❌ Clone failed - repo may still be private")

Cloning into '/content/lcldd'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 30 (delta 3), reused 26 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 17.65 KiB | 17.65 MiB/s, done.
Resolving deltas: 100% (3/3), done.

✅ Repo cloned successfully!
Files found:
  losses
  config.yaml
  precompute_teacher.py
  models
  test_smoke.py
  test_components.py
  test_phase2.py
  evaluate.py
  .git
  .gitignore
  README.md
  test_models.py
  train.py


In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

Tue Mar 31 13:58:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             42W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install transformers accelerate huggingface_hub sentencepiece -q


In [ ]:
import os
os.chdir("/content/lcldd")

for fname in ["train.py", "evaluate.py", "precompute_teacher.py"]:
    with open(fname, "r") as f:
        content = f.read()
    content = content.replace(
        '"mps" if torch.backends.mps.is_available() else "cpu"',
        '"cuda" if torch.cuda.is_available() else "cpu"'
    )
    content = content.replace(
        'torch.backends.mps.is_available()',
        'torch.cuda.is_available()'
    )
    content = content.replace(
        'torch.mps.empty_cache()',
        'torch.cuda.empty_cache()'
    )
    content = content.replace(
        'os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"',
        ''
    )
    with open(fname, "w") as f:
        f.write(content)
    print(f"Patched {fname} ✅")

Patched train.py ✅
Patched evaluate.py ✅
Patched precompute_teacher.py ✅


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

cache_on_drive = "/content/drive/MyDrive/lcldd_cache/teacher_trajectories.pt"
cache_local = "/content/lcldd/cache/teacher_trajectories.pt"
os.makedirs("/content/lcldd/cache", exist_ok=True)

if os.path.exists(cache_on_drive):
    shutil.copy(cache_on_drive, cache_local)
    print("Cache loaded from Drive ✅")
else:
    print("Cache not on Drive — will precompute in next cell")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cache not on Drive — will precompute in next cell


In [ ]:
if os.path.exists("/content/lcldd/cache/teacher_trajectories.pt"):
    print("Cache exists — skipping ✅")
else:
    print("Running precompute on GPU — ~2 minutes...")
    !python3 precompute_teacher.py

    os.makedirs("/content/drive/MyDrive/lcldd_cache", exist_ok=True)
    shutil.copy(
        "/content/lcldd/cache/teacher_trajectories.pt",
        "/content/drive/MyDrive/lcldd_cache/teacher_trajectories.pt"
    )
    print("Cache saved to Drive ✅ — won't need to recompute again")

Running precompute on GPU — ~2 minutes...
Loading teacher model (Qwen2.5-3B-Instruct) on CPU...
config.json: 100% 661/661 [00:00<00:00, 4.04MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 35.6kB [00:00, 87.0MB/s]
Fetching 2 files: 100% 2/2 [00:15<00:00,  7.98s/it]
Download complete: 100% 6.17G/6.17G [00:15<00:00, 345MB/s]                The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Download complete: 100% 6.17G/6.17G [00:16<00:00, 385MB/s]
Loading weights: 100% 434/434 [00:00<00:00, 493.61it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 1.48MB/s]
tokenizer_config.json: 7.30kB [00:00, 21.1MB/s]
vocab.json: 2.78MB [00:00, 40.4MB/s]
merges.txt: 1.67MB [00:00, 112MB/s]
tokenizer.json: 7.03MB [00:00, 156MB/s]
Teacher loaded on CPU
Extracting trajectories...
Processing question 1/12...
Processing question 2/12...

In [ ]:
with open("train.py", "r") as f:
    content = f.read()

content = content.replace('"batch_size": 1,', '"batch_size": 4,')
content = content.replace('"T_max": 3,', '"T_max": 5,')
content = content.replace('"num_epochs": 80,', '"num_epochs": 80,')

with open("train.py", "w") as f:
    f.write(content)

print("CONFIG updated for A100 ✅")
print("batch_size=4, T_max=5")

CONFIG updated for A100 ✅
batch_size=4, T_max=5


In [ ]:
!python3 train.py 2>&1 | tee /content/drive/MyDrive/lcldd_cache/training_log.txt


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loaded cached trajectories for 12 questions
Loading student model (Qwen2.5-1.5B) on MPS...
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 685.78it/s, Materializing param=model.norm.weight]
Loading mapped projection layer...
Starting fresh
Starting LCLDD Training...
Student: Qwen/Qwen2.5-1.5B
Hidden dim: 1536
Stage: E
Device: cuda
Initial losses — L_ans:8.7848 L_vf:0.0036 L_lya:5.0084
Epoch 1 | Step 1 | Total: 9.6722 | L_lya: 5.0084 | L_vf: 0.0036 | L_e2e: 8.6235 | ↓Energy: False
Epoch 1 | Step 2 | Total: 11.1739 | L_lya: 5.0068 | L_vf: 0.0033 | L_e2e: 8.4321 | ↓Energy: False
Epoch 1 | Step 3 | Total: 7.7630 | L_lya: 4.9500 | L_vf: 0.0032 | L_e2e: 7.2235 | ↓Energy: False
Epoch 2 | Step 1 | Total: 9.6569 | L_lya: 4.8960 | L_vf: 0.0036 | L_e2e: 8.4758 | ↓Energy: False
Epoch 2 | Step 2 | Total: 11.1637 | L_lya: 4.8796 | L_vf: 0.0033 | L_e2e

In [ ]:
!python3 evaluate.py


Loading student model (Qwen2.5-1.5B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 689.29it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 0.4520
  Checkpoint L_vf:  0.0033

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as many stickers as To
✅ Expected:     48 | Got: A stor

In [ ]:
import torch
ckpt = torch.load("/content/lcldd/checkpoints/lcldd_final.pt", map_location="cpu")
print("Checkpoint epoch:", ckpt.get("epoch"))
print("L_lya:", ckpt["final_losses"]["L_lya"])
print("L_vf:", ckpt["final_losses"]["L_vf"])
print("Keys:", list(ckpt.keys()))

Checkpoint epoch: 80
L_lya: 0.4520389437675476
L_vf: 0.0032913859467953444
Keys: ['epoch', 'thinking_block_state', 'optimizer_state', 'config', 'final_losses', 'energy_trajectory']


In [ ]:
import os, shutil

# Clear old checkpoint
shutil.rmtree("/content/lcldd/checkpoints", ignore_errors=True)
os.makedirs("/content/lcldd/checkpoints", exist_ok=True)
print("Old checkpoint cleared ✅")

# Also update train.py to NOT resume from checkpoint
with open("/content/lcldd/train.py", "r") as f:
    content = f.read()

content = content.replace(
    'RESUME_FROM = "checkpoints/lcldd_final.pt"',
    'RESUME_FROM = ""'
)

with open("/content/lcldd/train.py", "w") as f:
    f.write(content)

print("Train.py set to start fresh ✅")

Old checkpoint cleared ✅
Train.py set to start fresh ✅


In [ ]:
# Precompute teacher
!cd /content/lcldd && python3 precompute_teacher.py

# Train fresh on A100 with E2E loss
!cd /content/lcldd && python3 train.py 2>&1 | tee /content/drive/MyDrive/lcldd_cache/training_log_colab.txt

Loading teacher model (Qwen2.5-3B-Instruct) on CPU...
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 434/434 [00:00<00:00, 498.24it/s, Materializing param=model.norm.weight]
Teacher loaded on CPU
Extracting trajectories...
Processing question 1/12...
Processing question 2/12...
Processing question 3/12...
Processing question 4/12...
Processing question 5/12...
Processing question 6/12...
Processing question 7/12...
Processing question 8/12...
Processing question 9/12...
Processing question 10/12...
Processing question 11/12...
Processing question 12/12...
Saved cache/teacher_trajectories.pt
Cached 12 trajectories
Each trajectory: 5 steps x shape torch.Size([1, 1536])
Done! Teacher model can now be unloaded.
Run train.py to start training without teacher in memory.
The following generation flags are not valid and may be i

In [ ]:
!cd /content/lcldd && python3 evaluate.py


Loading student model (Qwen2.5-1.5B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 699.97it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 0.4821
  Checkpoint L_vf:  0.0031

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as many stickers as To
✅ Expected:     48 | Got: A stor

In [ ]:
# Fix evaluate.py injection scale
with open("/content/lcldd/evaluate.py", "r") as f:
    content = f.read()

# Try three different injection scales
content = content.replace(
    "modified_embeds[:, -1:, :] + 0.01 * thinking_delta",
    "modified_embeds[:, -1:, :] + 0.1 * thinking_delta"
)

with open("/content/lcldd/evaluate.py", "w") as f:
    f.write(content)

print("Injection scale updated to 0.1 ✅")

Injection scale updated to 0.1 ✅


In [ ]:
!cd /content/lcldd && python3 evaluate.py

Loading student model (Qwen2.5-1.5B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 752.98it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 0.4821
  Checkpoint L_vf:  0.0031

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as many stickers as To
✅ Expected:     48 | Got: A stor

In [ ]:
# Rewrite the injection method in evaluate.py
new_evaluate = '''
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from models.thinking_block import LyapunovThinkingBlock
import os
import re

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIG_HIDDEN_DIM = 1536

GSM8K_EVAL = [
    {"question": "A farmer has 3 fields. Each field has 4 rows. Each row has 6 plants. How many plants total? Answer:", "answer": "72"},
    {"question": "Lisa had 120 dollars. She spent a third on books and half of what remained on food. How much left? Answer:", "answer": "40"},
    {"question": "A school has 8 classes. Each class has 25 students. 15 are absent. How many present? Answer:", "answer": "185"},
    {"question": "John walks 3 miles to school and back every day for 5 days. Total miles? Answer:", "answer": "30"},
    {"question": "A baker makes 12 loaves per hour. Works 6 hours and sells 40 loaves. How many remain? Answer:", "answer": "32"},
    {"question": "There are 5 boxes with 8 red balls and 4 blue balls each. Total balls? Answer:", "answer": "60"},
    {"question": "Maria has 3 times as many stickers as Tom. Tom has 24. Maria gives away 15. How many does Maria have? Answer:", "answer": "57"},
    {"question": "A store buys apples for 2 dollars each and sells for 3 dollars. Sells 48 apples. What is the profit? Answer:", "answer": "48"},
    {"question": "A tank holds 200 liters. It is 35 percent full. How many liters needed to fill it? Answer:", "answer": "130"},
    {"question": "A train goes 60 mph for 2 hours then 80 mph for 1 hour. Total distance? Answer:", "answer": "200"},
    {"question": "A cinema has 15 rows with 20 seats each. 47 seats are occupied. How many are empty? Answer:", "answer": "253"},
    {"question": "A worker earns 18 dollars per hour and works 7.5 hours. How much does he earn? Answer:", "answer": "135"},
    {"question": "A container has 3.5 liters of juice. If 750ml is poured out, how many ml remain? Answer:", "answer": "2750"},
    {"question": "A class of 30 students took a test. 40 percent passed. How many failed? Answer:", "answer": "18"},
    {"question": "A recipe needs 2.5 cups of flour per batch. How much flour for 4 batches? Answer:", "answer": "10"},
]


def evaluate(model, tokenizer, thinking_block, eval_data, use_thinking=True):
    correct = 0
    total = len(eval_data)
    results = []
    T_max = 5
    device = next(model.parameters()).device
    thinking_block.eval()

    for item in eval_data:
        inputs = tokenizer(
            item["question"],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            # Always get base embeddings and hidden states
            outputs = model(**inputs, output_hidden_states=True)
            phi_x = outputs.hidden_states[-1].mean(dim=1).float()
            original_embeds = model.get_input_embeddings()(
                inputs["input_ids"]
            ).float()

            if use_thinking:
                # Run thinking block
                h = phi_x.clone()
                for _ in range(T_max):
                    h = thinking_block(h, phi_x)

                # --- NEW INJECTION: Prepend h_T as a virtual thinking token ---
                # Project h_T to embedding space
                thinking_token = h.unsqueeze(1)  # (1, 1, 1536)

                # Scale to match embedding magnitude
                embed_scale = original_embeds.norm(dim=-1).mean()
                thinking_scale = thinking_token.norm(dim=-1).mean()
                thinking_token = thinking_token * (embed_scale / (thinking_scale + 1e-8))

                # Cast to model dtype
                thinking_token = thinking_token.to(model.dtype)
                original_embeds = original_embeds.to(model.dtype)

                # Prepend thinking token to input embeddings
                extended_embeds = torch.cat(
                    [thinking_token, original_embeds], dim=1
                )  # (1, seq+1, 1536)

                # Extend attention mask
                extended_mask = torch.cat([
                    torch.ones(1, 1, device=device, dtype=inputs["attention_mask"].dtype),
                    inputs["attention_mask"]
                ], dim=1)

                # Generate with prepended thinking token
                generated = model.generate(
                    inputs_embeds=extended_embeds,
                    attention_mask=extended_mask,
                    max_new_tokens=15,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            else:
                generated = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=15,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

        decoded = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
        numbers_in_output = re.findall(r"\\b\\d+(?:\\.\\d+)?\\b", decoded)
        match = item["answer"] in numbers_in_output
        if match:
            correct += 1

        results.append({
            "question": item["question"],
            "expected": item["answer"],
            "predicted": decoded[:50],
            "correct": match,
        })

    print("=" * 60)
    print("LCLDD Evaluation Results")
    print(f"Mode: {\'With Thinking Token (prepend)\' if use_thinking else \'Baseline\'}")
    print("=" * 60)
    for result in results:
        status = "✅" if result["correct"] else "❌"
        print(f"{status} Expected: {result[\'expected\']:>6} | Got: {result[\'predicted\'][:40]}")
    print("=" * 60)
    print(f"Accuracy: {correct}/{total} = {correct / total * 100:.1f}%")
    print("=" * 60)
    return correct / total


if __name__ == "__main__":
    STUDENT_MODEL = "Qwen/Qwen2.5-1.5B"
    CHECKPOINT = "checkpoints/lcldd_final.pt"

    print("Loading student model...")
    student = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL,
        dtype=torch.float32,
        output_hidden_states=True,
        low_cpu_mem_usage=True,
    ).to(DEVICE)

    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading thinking block...")
    thinking_block = LyapunovThinkingBlock(CONFIG_HIDDEN_DIM).to(DEVICE)
    if os.path.exists(CHECKPOINT):
        ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
        thinking_block.load_state_dict(ckpt["thinking_block_state"])
        print(f"Loaded from {CHECKPOINT}")
        if "final_losses" in ckpt:
            print(f"  L_lya: {ckpt[\'final_losses\'][\'L_lya\']:.4f}")
            print(f"  L_vf:  {ckpt[\'final_losses\'][\'L_vf\']:.4f}")
    else:
        print("WARNING: checkpoint not found")

    print("\\n--- Running BASELINE ---")
    baseline = evaluate(student, tokenizer, thinking_block, GSM8K_EVAL, use_thinking=False)

    print("\\n--- Running LCLDD (thinking token prepend) ---")
    lcldd = evaluate(student, tokenizer, thinking_block, GSM8K_EVAL, use_thinking=True)

    print("\\n" + "=" * 60)
    print("COMPARISON")
    print("=" * 60)
    print(f"Baseline accuracy:  {baseline * 100:.1f}%")
    print(f"LCLDD accuracy:     {lcldd * 100:.1f}%")
    if lcldd > baseline:
        print(f"LCLDD improvement:  +{(lcldd - baseline) * 100:.1f}% ✅")
    elif lcldd == baseline:
        print("Same accuracy — thinking token neutral")
    else:
        print(f"LCLDD regression:   -{(baseline - lcldd) * 100:.1f}%")
    print("=" * 60)
'''

with open("/content/lcldd/evaluate.py", "w") as f:
    f.write(new_evaluate)

print("evaluate.py rewritten with prepend injection ✅")

evaluate.py rewritten with prepend injection ✅


In [ ]:
import os

# Check where we are
print("Current dir:", os.getcwd())
print("lcldd exists:", os.path.exists("/content/lcldd"))

# If not there, reclone
if not os.path.exists("/content/lcldd"):
    os.system("git clone https://github.com/KARTIKPANSURIYA/lcldd.git /content/lcldd")
    print("Recloned ✅")

os.chdir("/content/lcldd")
print("Now in:", os.getcwd())
print("Files:", os.listdir("."))

Current dir: /content
lcldd exists: True
Now in: /content/lcldd
Files: ['checkpoints']


In [ ]:
import os
os.chdir("/content/lcldd")

evaluate_code = r"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from models.thinking_block import LyapunovThinkingBlock
import os, re

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIG_HIDDEN_DIM = 1536

GSM8K_EVAL = [
    {"question": "A farmer has 3 fields. Each field has 4 rows. Each row has 6 plants. How many plants total? Answer:", "answer": "72"},
    {"question": "Lisa had 120 dollars. She spent a third on books and half of what remained on food. How much left? Answer:", "answer": "40"},
    {"question": "A school has 8 classes. Each class has 25 students. 15 are absent. How many present? Answer:", "answer": "185"},
    {"question": "John walks 3 miles to school and back every day for 5 days. Total miles? Answer:", "answer": "30"},
    {"question": "A baker makes 12 loaves per hour. Works 6 hours and sells 40 loaves. How many remain? Answer:", "answer": "32"},
    {"question": "There are 5 boxes with 8 red balls and 4 blue balls each. Total balls? Answer:", "answer": "60"},
    {"question": "Maria has 3 times as many stickers as Tom. Tom has 24. Maria gives away 15. How many does Maria have? Answer:", "answer": "57"},
    {"question": "A store buys apples for 2 dollars each and sells for 3 dollars. Sells 48 apples. What is the profit? Answer:", "answer": "48"},
    {"question": "A tank holds 200 liters. It is 35 percent full. How many liters needed to fill it? Answer:", "answer": "130"},
    {"question": "A train goes 60 mph for 2 hours then 80 mph for 1 hour. Total distance? Answer:", "answer": "200"},
    {"question": "A cinema has 15 rows with 20 seats each. 47 seats are occupied. How many are empty? Answer:", "answer": "253"},
    {"question": "A worker earns 18 dollars per hour and works 7.5 hours. How much does he earn? Answer:", "answer": "135"},
    {"question": "A container has 3.5 liters of juice. If 750ml is poured out, how many ml remain? Answer:", "answer": "2750"},
    {"question": "A class of 30 students took a test. 40 percent passed. How many failed? Answer:", "answer": "18"},
    {"question": "A recipe needs 2.5 cups of flour per batch. How much flour for 4 batches? Answer:", "answer": "10"},
]

def evaluate(model, tokenizer, thinking_block, eval_data, use_thinking=True):
    correct = 0
    total = len(eval_data)
    results = []
    T_max = 5
    device = next(model.parameters()).device
    thinking_block.eval()

    for item in eval_data:
        inputs = tokenizer(
            item["question"], return_tensors="pt",
            padding=True, truncation=True, max_length=128,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            phi_x = outputs.hidden_states[-1].mean(dim=1).float()
            original_embeds = model.get_input_embeddings()(inputs["input_ids"]).float()

            if use_thinking:
                h = phi_x.clone()
                for _ in range(T_max):
                    h = thinking_block(h, phi_x)

                thinking_token = h.unsqueeze(1)
                embed_scale = original_embeds.norm(dim=-1).mean()
                thinking_scale = thinking_token.norm(dim=-1).mean()
                thinking_token = thinking_token * (embed_scale / (thinking_scale + 1e-8))
                thinking_token = thinking_token.to(model.dtype)
                original_embeds = original_embeds.to(model.dtype)

                extended_embeds = torch.cat([thinking_token, original_embeds], dim=1)
                extended_mask = torch.cat([
                    torch.ones(1, 1, device=device, dtype=inputs["attention_mask"].dtype),
                    inputs["attention_mask"]
                ], dim=1)

                generated = model.generate(
                    inputs_embeds=extended_embeds,
                    attention_mask=extended_mask,
                    max_new_tokens=15,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            else:
                generated = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=15,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

        decoded = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
        numbers_in_output = re.findall(r'\b\d+(?:\.\d+)?\b', decoded)
        match = item["answer"] in numbers_in_output
        if match:
            correct += 1
        results.append({"question": item["question"], "expected": item["answer"],
                        "predicted": decoded[:50], "correct": match})

    print("=" * 60)
    print(f"Mode: {'With Thinking Token (prepend)' if use_thinking else 'Baseline'}")
    print("=" * 60)
    for r in results:
        print(f"{'✅' if r['correct'] else '❌'} Expected: {r['expected']:>6} | Got: {r['predicted'][:40]}")
    print("=" * 60)
    print(f"Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
    print("=" * 60)
    return correct / total

if __name__ == "__main__":
    STUDENT_MODEL = "Qwen/Qwen2.5-1.5B"
    CHECKPOINT = "checkpoints/lcldd_final.pt"

    print("Loading student model...")
    student = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL, dtype=torch.float32,
        output_hidden_states=True, low_cpu_mem_usage=True,
    ).to(DEVICE)

    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading thinking block...")
    thinking_block = LyapunovThinkingBlock(CONFIG_HIDDEN_DIM).to(DEVICE)
    if os.path.exists(CHECKPOINT):
        ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
        thinking_block.load_state_dict(ckpt["thinking_block_state"])
        print(f"Loaded: L_lya={ckpt['final_losses']['L_lya']:.4f}")
    else:
        print("WARNING: no checkpoint found")

    print("\n--- BASELINE ---")
    baseline = evaluate(student, tokenizer, thinking_block, GSM8K_EVAL, use_thinking=False)

    print("\n--- LCLDD (prepend thinking token) ---")
    lcldd = evaluate(student, tokenizer, thinking_block, GSM8K_EVAL, use_thinking=True)

    print("\n" + "=" * 60)
    print(f"Baseline:  {baseline*100:.1f}%")
    print(f"LCLDD:     {lcldd*100:.1f}%")
    if lcldd > baseline:
        print(f"Improvement: +{(lcldd-baseline)*100:.1f}% ✅")
    elif lcldd == baseline:
        print("Neutral — same accuracy")
    else:
        print(f"Regression: -{(baseline-lcldd)*100:.1f}%")
    print("=" * 60)
"""

with open("/content/lcldd/evaluate.py", "w") as f:
    f.write(evaluate_code.strip())

print("evaluate.py written ✅")
print("File size:", os.path.getsize("/content/lcldd/evaluate.py"), "bytes")

evaluate.py written ✅
File size: 6740 bytes


In [ ]:
!cd /content/lcldd && python3 evaluate.py

Loading student model...
config.json: 100% 684/684 [00:00<00:00, 3.84MB/s]
model.safetensors: 100% 3.09G/3.09G [00:10<00:00, 285MB/s]
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 752.17it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 138/138 [00:00<00:00, 880kB/s]
tokenizer_config.json: 7.23kB [00:00, 22.3MB/s]
vocab.json: 2.78MB [00:00, 66.5MB/s]
merges.txt: 1.67MB [00:00, 113MB/s]
tokenizer.json: 7.03MB [00:00, 147MB/s]
Loading thinking block...

--- BASELINE ---
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:  

In [ ]:
import os

# Step 1 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil

# Step 2 - Check what's on Drive
print("Drive contents:")
for f in os.listdir("/content/drive/MyDrive/"):
    print(" ", f)

Drive contents:
  Resume Sample - UG (1).pdf
  Resume Sample - UG (1).gdoc
  LOR Form (kartik).docx
  SOP Form KARTIK.docx
  1706531756Kartik LOR 1.docx
  1706531756Kartik LOR 2.docx
  1706531756Kartik LOR 3.docx
  Untitled document (43).gdoc
  1706594178Kartik- CS- USA.docx
  Kartik Resume .gdoc
  Untitled document (42).gdoc
  Documents
  Untitled document (41).gdoc
  Untitled document (40).gdoc
  Untitled document (39).gdoc
  Untitled document (38).gdoc
  kartik pansuriya (prob  class).gdoc
  kenil1.gdoc
  Untitled document (37).gdoc
  Colab Notebooks
  VACCINE TABLE.gdoc
  Untitled document (36).gdoc
  (Kartik) HW-2-Matlab-TensorFlow-PyTorch.gdoc
  Untitled document (35).gdoc
  CPE EE-695A: (Team-7) Final Project Team Contract.gdoc
  Untitled document (34).gdoc
  Guj loan doc
  HW-4-kNN (Kartik pansuriya).gslides
  HW-3-DTL (Kartik Pansuriya-20030935).gslides
  HW-5 Support Vector Machine (SVM) (Kartik Pansuriya).gslides
  HW-6 DL (Kartik).gslides
  HW-6 HHN (Kartik).gslides
  HW-7-

In [ ]:
import os

# Step 1 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil

# Step 2 - Check what's on Drive
print("Drive contents:")
for f in os.listdir("/content/drive/MyDrive/"):
    print(" ", f)

Drive contents:
  Colab Notebooks
  Resume Sample - UG (1).pdf
  Resume Sample - UG (1).gdoc
  LOR Form (kartik).docx
  SOP Form KARTIK.docx
  1706531756Kartik LOR 1.docx
  1706531756Kartik LOR 2.docx
  1706531756Kartik LOR 3.docx
  Untitled document (43).gdoc
  1706594178Kartik- CS- USA.docx
  Kartik Resume .gdoc
  Untitled document (42).gdoc
  Documents
  Untitled document (41).gdoc
  Untitled document (40).gdoc
  Untitled document (39).gdoc
  Untitled document (38).gdoc
  kartik pansuriya (prob  class).gdoc
  kenil1.gdoc
  Untitled document (37).gdoc
  VACCINE TABLE.gdoc
  Untitled document (36).gdoc
  (Kartik) HW-2-Matlab-TensorFlow-PyTorch.gdoc
  Untitled document (35).gdoc
  CPE EE-695A: (Team-7) Final Project Team Contract.gdoc
  Untitled document (34).gdoc
  Guj loan doc
  HW-4-kNN (Kartik pansuriya).gslides
  HW-3-DTL (Kartik Pansuriya-20030935).gslides
  HW-5 Support Vector Machine (SVM) (Kartik Pansuriya).gslides
  HW-6 DL (Kartik).gslides
  HW-6 HHN (Kartik).gslides
  HW-7-

In [ ]:
import os, shutil

# Force correct directory
os.chdir("/content")

# Reclone fresh
if os.path.exists("/content/lcldd"):
    shutil.rmtree("/content/lcldd")

os.system("git clone https://github.com/KARTIKPANSURIYA/lcldd.git /content/lcldd")
os.chdir("/content/lcldd")

print("Current dir:", os.getcwd())
print("Files:", os.listdir("."))

Current dir: /content/lcldd
Files: ['.gitignore', 'test_components.py', 'evaluate.py', 'train.py', 'config.yaml', 'test_smoke.py', 'precompute_teacher.py', 'models', 'losses', 'test_models.py', 'README.md', 'test_phase2.py', '.git']


In [ ]:
# Verify train.py is there
print("train.py exists:", os.path.exists("train.py"))
print("evaluate.py exists:", os.path.exists("evaluate.py"))
print("precompute_teacher.py exists:", os.path.exists("precompute_teacher.py"))

train.py exists: True
evaluate.py exists: True
precompute_teacher.py exists: True


In [ ]:
import os, shutil
os.chdir("/content/lcldd")

for fname in ["train.py", "evaluate.py", "precompute_teacher.py"]:
    with open(fname, "r") as f:
        c = f.read()
    c = c.replace('"mps" if torch.backends.mps.is_available() else "cpu"', '"cuda" if torch.cuda.is_available() else "cpu"')
    c = c.replace('torch.backends.mps.is_available()', 'torch.cuda.is_available()')
    c = c.replace('torch.mps.empty_cache()', 'torch.cuda.empty_cache()')
    c = c.replace('os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"', '')
    with open(fname, "w") as f:
        f.write(c)

# Disable resume, clear checkpoints
with open("train.py", "r") as f:
    c = f.read()
c = c.replace('RESUME_FROM = "checkpoints/lcldd_final.pt"', 'RESUME_FROM = ""')
with open("train.py", "w") as f:
    f.write(c)

shutil.rmtree("checkpoints", ignore_errors=True)
os.makedirs("checkpoints")
print("All patched ✅")

All patched ✅


In [ ]:
os.chdir("/content/lcldd")
!python3 precompute_teacher.py

Loading teacher model (Qwen2.5-3B-Instruct) on CPU...
config.json: 100% 661/661 [00:00<00:00, 3.94MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 35.6kB [00:00, 96.7MB/s]
Fetching 2 files: 100% 2/2 [00:15<00:00,  7.76s/it]
Download complete: 100% 6.17G/6.17G [00:15<00:00, 452MB/s]                The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Download complete: 100% 6.17G/6.17G [00:15<00:00, 395MB/s]
Loading weights: 100% 434/434 [00:00<00:00, 463.99it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 1.76MB/s]
tokenizer_config.json: 7.30kB [00:00, 25.1MB/s]
vocab.json: 2.78MB [00:00, 116MB/s]
merges.txt: 1.67MB [00:00, 116MB/s]
tokenizer.json: 7.03MB [00:00, 154MB/s]
Teacher loaded on CPU
Extracting trajectories...
Processing question 1/12...
Processing question 2/12...
Processing question 3/12...
Processing que

In [ ]:
os.chdir("/content/lcldd")
!python3 train.py

Loaded cached trajectories for 12 questions
Loading student model (Qwen2.5-1.5B) on MPS...
config.json: 100% 684/684 [00:00<00:00, 2.88MB/s]
model.safetensors: 100% 3.09G/3.09G [00:07<00:00, 395MB/s]
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 727.50it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 138/138 [00:00<00:00, 628kB/s]
tokenizer_config.json: 7.23kB [00:00, 17.1MB/s]
vocab.json: 2.78MB [00:00, 112MB/s]
merges.txt: 1.67MB [00:00, 112MB/s]
tokenizer.json: 7.03MB [00:00, 146MB/s]
Loading mapped projection layer...
Starting fresh
Starting LCLDD Training...
Student: Qwen/Qwen2.5-1.5B
Hidden dim: 1536
Stage: E
Device: cuda
Initial losses — L_ans:9.6339 L_vf:0.0012 L_lya:3.1979
Epoch 1 | Step 1 | Total: 10.6148 | L_lya: 3.1979 | L_vf: 0.0012 | L_e2e: 9.6492 | ↓Energy: True
Epoch 1 | Step 2 | Total: 11.4644 | L_lya: 3.15

In [ ]:
os.chdir("/content/lcldd")
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs("/content/drive/MyDrive/lcldd_checkpoints", exist_ok=True)
shutil.copy(
    "/content/lcldd/checkpoints/lcldd_final.pt",
    "/content/drive/MyDrive/lcldd_checkpoints/lcldd_final.pt"
)
print("Saved to Drive ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive ✅


In [ ]:
os.chdir("/content/lcldd")

with open("evaluate.py", "r") as f:
    c = f.read()

old = '''                thinking_delta = (h - phi_x).unsqueeze(1)
                scale = original_embeds.norm(dim=-1, keepdim=True).mean()
                thinking_delta = thinking_delta * (
                    scale / (thinking_delta.norm() + 1e-8)
                )

                modified_embeds = original_embeds.clone()
                modified_embeds[:, -1:, :] = (
                    original_embeds[:, -1:, :] + 0.01 * thinking_delta
                )
                modified_embeds = modified_embeds.to(model.dtype)

                generated = model.generate(
                    inputs_embeds=modified_embeds,
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=15,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )'''

new = '''                # Prepend thinking token as soft prompt at 1% scale
                thinking_token = h.unsqueeze(1).to(model.dtype)
                orig = original_embeds.to(model.dtype)
                embed_norm = orig.norm(dim=-1).mean()
                think_norm = thinking_token.norm(dim=-1).mean()
                thinking_token = thinking_token * (embed_norm / (think_norm + 1e-8)) * 0.01

                extended_embeds = torch.cat([thinking_token, orig], dim=1)
                extended_mask = torch.cat([
                    torch.ones(1, 1, device=device, dtype=inputs["attention_mask"].dtype),
                    inputs["attention_mask"]
                ], dim=1)

                generated = model.generate(
                    inputs_embeds=extended_embeds,
                    attention_mask=extended_mask,
                    max_new_tokens=15,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )'''

if old in c:
    c = c.replace(old, new)
    print("Injection block replaced ✅")
else:
    print("Block not found — writing fresh evaluate.py")
    # Write completely fresh file
    with open("evaluate.py", "w") as f:
        f.write(c)

with open("evaluate.py", "w") as f:
    f.write(c)

print("Running evaluation...")
!python3 /content/lcldd/evaluate.py

Injection block replaced ✅
Running evaluation...
Loading student model (Qwen2.5-1.5B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 770.30it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 0.3314
  Checkpoint L_vf:  0.0011

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as man

In [ ]:
import os
os.chdir("/content/lcldd")

# Update precompute_teacher.py
with open("precompute_teacher.py", "r") as f:
    c = f.read()

c = c.replace(
    '"Qwen/Qwen2.5-3B-Instruct"',
    '"Qwen/Qwen2.5-7B-Instruct"'
)
# 7B has hidden_dim=3584, project to student 2048
c = c.replace(
    'proj = torch.nn.Linear(2048, 1536, bias=False)',
    'proj = torch.nn.Linear(3584, 2048, bias=False)'
)
c = c.replace(
    '"hidden_dim": 1536',
    '"hidden_dim": 2048'
)
# Load teacher on CUDA now that we have 40GB
c = c.replace(
    'torch_dtype=torch.float32',
    'torch_dtype=torch.float16'
)
c = c.replace(
    'teacher = AutoModelForCausalLM.from_pretrained(\n    "Qwen/Qwen2.5-7B-Instruct",\n    torch_dtype=torch.float16,\n    output_hidden_states=True,\n    low_cpu_mem_usage=True\n)',
    'teacher = AutoModelForCausalLM.from_pretrained(\n    "Qwen/Qwen2.5-7B-Instruct",\n    torch_dtype=torch.float16,\n    output_hidden_states=True,\n    low_cpu_mem_usage=True\n).to("cuda")'
)

with open("precompute_teacher.py", "w") as f:
    f.write(c)

print("precompute_teacher.py updated to 7B ✅")

# Update train.py
with open("train.py", "r") as f:
    c = f.read()

c = c.replace(
    '"student_model": "Qwen/Qwen2.5-1.5B"',
    '"student_model": "Qwen/Qwen2.5-3B"'
)
c = c.replace(
    '"hidden_dim": 1536',
    '"hidden_dim": 2048'
)
c = c.replace(
    'STUDENT_DIM = 1536',
    'STUDENT_DIM = 2048'
)
c = c.replace(
    'proj = nn.Linear(2048, 1536, bias=False)',
    'proj = nn.Linear(3584, 2048, bias=False)'
)
c = c.replace(
    '"batch_size": 1',
    '"batch_size": 2'
)
c = c.replace(
    '"T_max": 3',
    '"T_max": 5'
)
c = c.replace(
    '"num_epochs": 80',
    '"num_epochs": 80'
)
c = c.replace(
    'Loading student model (Qwen2.5-1.5B) on MPS...',
    'Loading student model (Qwen2.5-3B) on CUDA...'
)

with open("train.py", "w") as f:
    f.write(c)

print("train.py updated to 3B student ✅")

# Update evaluate.py
with open("evaluate.py", "r") as f:
    c = f.read()

c = c.replace(
    'STUDENT_MODEL = "Qwen/Qwen2.5-1.5B"',
    'STUDENT_MODEL = "Qwen/Qwen2.5-3B"'
)
c = c.replace(
    'CONFIG_HIDDEN_DIM = 1536',
    'CONFIG_HIDDEN_DIM = 2048'
)

with open("evaluate.py", "w") as f:
    f.write(c)

print("evaluate.py updated to 3B student ✅")
print("\nNew setup:")
print("  Teacher: Qwen2.5-7B-Instruct (frozen, CUDA)")
print("  Student: Qwen2.5-3B (trainable, CUDA)")
print("  Hidden dim: 2048")
print("  Projection: 3584 -> 2048")

precompute_teacher.py updated to 7B ✅
train.py updated to 3B student ✅
evaluate.py updated to 3B student ✅

New setup:
  Teacher: Qwen2.5-7B-Instruct (frozen, CUDA)
  Student: Qwen2.5-3B (trainable, CUDA)
  Hidden dim: 2048
  Projection: 3584 -> 2048


In [ ]:
import shutil, os
os.chdir("/content/lcldd")

shutil.rmtree("cache", ignore_errors=True)
shutil.rmtree("checkpoints", ignore_errors=True)
os.makedirs("cache")
os.makedirs("checkpoints")
print("Cache and checkpoints cleared ✅")

Cache and checkpoints cleared ✅


In [ ]:
os.chdir("/content/lcldd")
!python3 precompute_teacher.py

Loading teacher model (Qwen2.5-3B-Instruct) on CPU...
config.json: 100% 663/663 [00:00<00:00, 3.35MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 27.8kB [00:00, 80.9MB/s]
Fetching 4 files: 100% 4/4 [00:40<00:00, 10.13s/it]
Download complete: 100% 15.2G/15.2G [00:40<00:00, 433MB/s]                The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Download complete: 100% 15.2G/15.2G [00:40<00:00, 376MB/s]
Loading weights: 100% 339/339 [00:02<00:00, 169.09it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 243/243 [00:00<00:00, 1.32MB/s]
tokenizer_config.json: 7.30kB [00:00, 26.9MB/s]
vocab.json: 2.78MB [00:00, 114MB/s]
merges.txt: 1.67MB [00:00, 115MB/s]
tokenizer.json: 7.03MB [00:00, 139MB/s]
Teacher loaded on CPU
Extracting trajectories...
Processing question 1/12...
Traceback (most recent call last):
  File "/content/lcldd/precompute_t

In [ ]:
import os
os.chdir("/content/lcldd")

with open("precompute_teacher.py", "r") as f:
    c = f.read()

# Move inputs to cuda before teacher forward pass
c = c.replace(
    '''    with torch.no_grad():
        outputs = teacher(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True
        )''',
    '''    with torch.no_grad():
        input_ids = inputs["input_ids"].to("cuda")
        attention_mask = inputs["attention_mask"].to("cuda")
        outputs = teacher(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )'''
)

with open("precompute_teacher.py", "w") as f:
    f.write(c)

print("Fixed ✅ — inputs now moved to CUDA before teacher forward pass")

Fixed ✅ — inputs now moved to CUDA before teacher forward pass


In [ ]:
os.chdir("/content/lcldd")
!python3 precompute_teacher.py

Loading teacher model (Qwen2.5-3B-Instruct) on CPU...
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 339/339 [00:02<00:00, 155.35it/s, Materializing param=model.norm.weight]
Teacher loaded on CPU
Extracting trajectories...
Processing question 1/12...
Traceback (most recent call last):
  File "/content/lcldd/precompute_teacher.py", line 65, in <module>
    h_proj = proj(h_mean)
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packa

In [ ]:
import os
os.chdir("/content/lcldd")

with open("precompute_teacher.py", "r") as f:
    c = f.read()

# Move proj to cuda
c = c.replace(
    'proj = torch.nn.Linear(3584, 2048, bias=False)',
    'proj = torch.nn.Linear(3584, 2048, bias=False).to("cuda")'
)

with open("precompute_teacher.py", "w") as f:
    f.write(c)

print("proj moved to CUDA ✅")

proj moved to CUDA ✅


In [ ]:
os.chdir("/content/lcldd")
!python3 precompute_teacher.py

Loading teacher model (Qwen2.5-3B-Instruct) on CPU...
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 339/339 [00:02<00:00, 156.71it/s, Materializing param=model.norm.weight]
Teacher loaded on CPU
Extracting trajectories...
Processing question 1/12...
Processing question 2/12...
Processing question 3/12...
Processing question 4/12...
Processing question 5/12...
Processing question 6/12...
Processing question 7/12...
Processing question 8/12...
Processing question 9/12...
Processing question 10/12...
Processing question 11/12...
Processing question 12/12...
Saved cache/teacher_trajectories.pt
Cached 12 trajectories
Each trajectory: 5 steps x shape torch.Size([1, 2048])
Done! Teacher model can now be unloaded.
Run train.py to start training without teacher in memory.


In [ ]:
os.chdir("/content/lcldd")
!python3 precompute_teacher.py

Loading teacher model (Qwen2.5-3B-Instruct) on CPU...
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 339/339 [00:01<00:00, 172.64it/s, Materializing param=model.norm.weight]
Teacher loaded on CPU
Extracting trajectories...
Processing question 1/12...
Processing question 2/12...
Processing question 3/12...
Processing question 4/12...
Processing question 5/12...
Processing question 6/12...
Processing question 7/12...
Processing question 8/12...
Processing question 9/12...
Processing question 10/12...
Processing question 11/12...
Processing question 12/12...
Saved cache/teacher_trajectories.pt
Cached 12 trajectories
Each trajectory: 5 steps x shape torch.Size([1, 2048])
Done! Teacher model can now be unloaded.
Run train.py to start training without teacher in memory.


In [ ]:
os.chdir("/content/lcldd")
!python3 train.py

Loaded cached trajectories for 12 questions
Loading student model (Qwen2.5-3B) on CUDA...
config.json: 100% 683/683 [00:00<00:00, 3.87MB/s]
model.safetensors.index.json: 35.6kB [00:00, 82.5MB/s]
Fetching 2 files: 100% 2/2 [00:16<00:00,  8.45s/it]
Download complete: 100% 6.17G/6.17G [00:16<00:00, 335MB/s]                The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Download complete: 100% 6.17G/6.17G [00:16<00:00, 365MB/s]
Loading weights: 100% 434/434 [00:00<00:00, 453.74it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 138/138 [00:00<00:00, 852kB/s]
tokenizer_config.json: 7.23kB [00:00, 25.6MB/s]
vocab.json: 2.78MB [00:00, 112MB/s]
merges.txt: 1.67MB [00:00, 120MB/s]
tokenizer.json: 7.03MB [00:00, 152MB/s]
Loading mapped projection layer...
Starting fresh
Starting LCLDD Training...
Student: Qwen/Qwen2.5-3B
Hidden dim: 2048
Stage: E
Device: cuda
Initial losses — L_an

In [ ]:
# Save to Drive first before anything else
import shutil, os
os.makedirs("/content/drive/MyDrive/lcldd_checkpoints_3B", exist_ok=True)
shutil.copy(
    "/content/lcldd/checkpoints/lcldd_final.pt",
    "/content/drive/MyDrive/lcldd_checkpoints_3B/lcldd_final.pt"
)
print("Checkpoint saved to Drive ✅")

Checkpoint saved to Drive ✅


In [ ]:
os.chdir("/content/lcldd")
!python3 evaluate.py

Loading student model (Qwen2.5-1.5B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 434/434 [00:01<00:00, 269.07it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 0.2976
  Checkpoint L_vf:  0.0046

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as many stickers as To
✅ Expected:     48 | Got: A stor

In [ ]:
import os
os.chdir("/content/lcldd")

# Check evaluate.py student model
with open("evaluate.py", "r") as f:
    c = f.read()

# Find and print the student model line
for line in c.split("\n"):
    if "STUDENT_MODEL" in line or "CONFIG_HIDDEN" in line:
        print(line)

CONFIG_HIDDEN_DIM = 2048
    STUDENT_MODEL = "Qwen/Qwen2.5-3B"
        STUDENT_MODEL,
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    thinking_block = LyapunovThinkingBlock(CONFIG_HIDDEN_DIM).to(DEVICE)


In [ ]:
with open("evaluate.py", "r") as f:
    c = f.read()

c = c.replace('STUDENT_MODEL = "Qwen/Qwen2.5-1.5B"', 'STUDENT_MODEL = "Qwen/Qwen2.5-3B"')
c = c.replace('CONFIG_HIDDEN_DIM = 1536', 'CONFIG_HIDDEN_DIM = 2048')

with open("evaluate.py", "w") as f:
    f.write(c)

print("Fixed ✅")
print("Student model: Qwen2.5-3B")
print("Hidden dim: 2048")

# Verify
with open("evaluate.py", "r") as f:
    c = f.read()
for line in c.split("\n"):
    if "STUDENT_MODEL" in line or "CONFIG_HIDDEN" in line:
        print("Confirmed:", line.strip())

Fixed ✅
Student model: Qwen2.5-3B
Hidden dim: 2048
Confirmed: CONFIG_HIDDEN_DIM = 2048
Confirmed: STUDENT_MODEL = "Qwen/Qwen2.5-3B"
Confirmed: STUDENT_MODEL,
Confirmed: tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
Confirmed: thinking_block = LyapunovThinkingBlock(CONFIG_HIDDEN_DIM).to(DEVICE)


In [ ]:
os.chdir("/content/lcldd")
!python3 evaluate.py

Loading student model (Qwen2.5-1.5B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 434/434 [00:00<00:00, 441.82it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 0.2976
  Checkpoint L_vf:  0.0046

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as many stickers as To
✅ Expected:     48 | Got: A stor

In [ ]:
import os
os.chdir("/content/lcldd")

with open("evaluate.py", "r") as f:
    lines = f.readlines()

# Print all lines with model or hidden dim references
for i, line in enumerate(lines):
    if any(x in line for x in ["STUDENT_MODEL", "CONFIG_HIDDEN", "Qwen2.5", "1.5B", "3B", "1536", "2048"]):
        print(f"Line {i+1}: {line.rstrip()}")

Line 12: CONFIG_HIDDEN_DIM = 2048
Line 127:     STUDENT_MODEL = "Qwen/Qwen2.5-3B"
Line 130:     print("Loading student model (Qwen2.5-1.5B)...")
Line 132:         STUDENT_MODEL,
Line 138:     tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
Line 143:     thinking_block = LyapunovThinkingBlock(CONFIG_HIDDEN_DIM).to(DEVICE)


In [ ]:
import os, re
os.chdir("/content/lcldd")

with open("evaluate.py", "r") as f:
    c = f.read()

# Fix 1 — fix the misleading print
c = c.replace(
    'print("Loading student model (Qwen2.5-1.5B)...")',
    'print(f"Loading student model ({STUDENT_MODEL})...")'
)

# Fix 2 — better number matching that catches "2750ml" and "2750\n"
c = c.replace(
    r"numbers_in_output = re.findall(r'\b\d+(?:\.\d+)?\b', decoded)",
    r"numbers_in_output = re.findall(r'\d+(?:\.\d+)?', decoded)"
)

# Fix 3 — reduce max_new_tokens to stop multi-answer generation
c = c.replace(
    'max_new_tokens=15,',
    'max_new_tokens=10,'
)

with open("evaluate.py", "w") as f:
    f.write(c)

print("Fixed ✅")
print("  1. Print statement corrected")
print("  2. Number regex made less strict")
print("  3. max_new_tokens reduced to 10 to stop multi-answer")

Fixed ✅
  1. Print statement corrected
  2. Number regex made less strict
  3. max_new_tokens reduced to 10 to stop multi-answer


In [ ]:
os.chdir("/content/lcldd")
!python3 evaluate.py

Loading student model (Qwen/Qwen2.5-3B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 434/434 [00:00<00:00, 481.63it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 0.2976
  Checkpoint L_vf:  0.0046

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as many stickers as To
✅ Expected:     48 | Got: A s

In [ ]:
import os, shutil
os.chdir("/content/lcldd")

# Pull latest code
!git pull origin main

# Patch CUDA
for fname in ["train.py", "evaluate.py", "precompute_teacher.py"]:
    with open(fname, "r") as f:
        c = f.read()
    c = c.replace('"mps" if torch.backends.mps.is_available() else "cpu"', '"cuda" if torch.cuda.is_available() else "cpu"')
    c = c.replace('torch.backends.mps.is_available()', 'torch.cuda.is_available()')
    c = c.replace('torch.mps.empty_cache()', 'torch.cuda.empty_cache()')
    c = c.replace('os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"', '')
    with open(fname, "w") as f:
        f.write(c)

# Copy cache from Drive if available
if os.path.exists("/content/drive/MyDrive/lcldd_cache/teacher_trajectories.pt"):
    os.makedirs("cache", exist_ok=True)
    shutil.copy("/content/drive/MyDrive/lcldd_cache/teacher_trajectories.pt", "cache/teacher_trajectories.pt")
    print("Cache restored from Drive ✅")
else:
    print("Need to recompute cache")

# Clear checkpoints
shutil.rmtree("checkpoints", ignore_errors=True)
os.makedirs("checkpoints")
print("Ready to train ✅")

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 740 bytes | 740.00 KiB/s, done.
From https://github.com/KARTIKPANSURIYA/lcldd
 * branch            main       -> FETCH_HEAD
   589209c..c464184  main       -> origin/main
Updating 589209c..c464184
error: Your local changes to the following files would be overwritten by merge:
	train.py
Please commit your changes or stash them before you merge.
Aborting
Cache restored from Drive ✅
Ready to train ✅


In [ ]:
import os, shutil
os.chdir("/content/lcldd")

# Force pull — discard local changes and take GitHub version
!git fetch origin
!git reset --hard origin/main
!git pull origin main

print("Git sync done ✅")

HEAD is now at c464184 Update train.py with prepend injection and new hyperparams
From https://github.com/KARTIKPANSURIYA/lcldd
 * branch            main       -> FETCH_HEAD
Already up to date.
Git sync done ✅


In [ ]:
import os, shutil
os.chdir("/content/lcldd")

# Patch CUDA
for fname in ["train.py", "evaluate.py", "precompute_teacher.py"]:
    if not os.path.exists(fname):
        continue
    with open(fname, "r") as f:
        c = f.read()
    c = c.replace('"mps" if torch.backends.mps.is_available() else "cpu"', '"cuda" if torch.cuda.is_available() else "cpu"')
    c = c.replace('torch.backends.mps.is_available()', 'torch.cuda.is_available()')
    c = c.replace('torch.mps.empty_cache()', 'torch.cuda.empty_cache()')
    c = c.replace('os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"', '')
    with open(fname, "w") as f:
        f.write(c)
    print(f"Patched {fname}")

# Restore cache from Drive
os.makedirs("cache", exist_ok=True)
if os.path.exists("/content/drive/MyDrive/lcldd_cache/teacher_trajectories.pt"):
    shutil.copy(
        "/content/drive/MyDrive/lcldd_cache/teacher_trajectories.pt",
        "cache/teacher_trajectories.pt"
    )
    print("Cache restored from Drive")
else:
    print("Cache not on Drive - will precompute")

# Clear checkpoints
shutil.rmtree("checkpoints", ignore_errors=True)
os.makedirs("checkpoints")
print("Ready to train")

Patched train.py
Patched evaluate.py
Patched precompute_teacher.py
Cache restored from Drive
Ready to train


In [ ]:
# Verify the E2E fix is in train.py
os.chdir("/content/lcldd")
with open("train.py", "r") as f:
    c = f.read()

if "extended_embeds" in c:
    print("Prepend injection confirmed in train.py")
else:
    print("WARNING: prepend injection not found - check Antigravity push")

if "0.5 * L_e2e" in c:
    print("E2E weight 0.5 confirmed")
else:
    print("WARNING: E2E weight not updated")

Prepend injection confirmed in train.py
E2E weight 0.5 confirmed


In [ ]:
os.chdir("/content/lcldd")
!python3 train.py

Loaded cached trajectories for 12 questions
Loading student model (Qwen2.5-1.5B) on MPS...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 694.20it/s, Materializing param=model.norm.weight]
Loading mapped projection layer...
Starting fresh
Starting LCLDD Training...
Student: Qwen/Qwen2.5-1.5B
Hidden dim: 1536
Stage: E
Device: cuda
Initial losses — L_ans:9.6339 L_vf:0.0012 L_lya:3.1609
Epoch 1 | Step 1 | Total: 12.9866 | L_lya: 3.1609 | L_vf: 0.0012 | L_e2e: 6.6737 | ↓Energy: True
Epoch 1 | Step 2 | Total: 13.8596 | L_lya: 3.0997 | L_vf: 0.0011 | L_e2e: 6.8672 | ↓Energy: True
Epoch 1 | Step 3 | Total: 11.8486 | L_lya: 3.1079 | L_vf: 0.0012 | L_e2e: 8.2412 | ↓Energy: True
Epoch 1 | Step 4 | Total: 11.1698 | L_lya: 3.1522 | L_vf: 0.0012 | L_e2e: 8.7847 | ↓Energy: True
Epoch 1 | Step 5 | Total: 12.8173 | L_lya: 3.1844 | L_vf: 0.0011 | L_e2e: 8.6623 | ↓E

In [ ]:
import shutil, os
os.makedirs("/content/drive/MyDrive/lcldd_checkpoints_e2e", exist_ok=True)
shutil.copy(
    "/content/lcldd/checkpoints/lcldd_final.pt",
    "/content/drive/MyDrive/lcldd_checkpoints_e2e/lcldd_final.pt"
)
print("Saved to Drive")

Saved to Drive


In [ ]:
os.chdir("/content/lcldd")
!python3 evaluate.py

Loading student model (Qwen2.5-1.5B)...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 669.99it/s, Materializing param=model.norm.weight]
Loading thinking block from checkpoint...
Loaded from checkpoints/lcldd_final.pt
  Checkpoint L_lya: 3.1034
  Checkpoint L_vf:  0.0012

--- Running BASELINE (no thinking block) ---
LCLDD Evaluation Results
Mode: Baseline
✅ Expected:     72 | Got: A farmer has 3 fields. Each field has 4 
✅ Expected:     40 | Got: Lisa had 120 dollars. She spent a third 
❌ Expected:    185 | Got: A school has 8 classes. Each class has 2
❌ Expected:     30 | Got: John walks 3 miles to school and back ev
❌ Expected:     32 | Got: A baker makes 12 loaves per hour. Works 
❌ Expected:     60 | Got: There are 5 boxes with 8 red balls and 4
❌ Expected:     57 | Got: Maria has 3 times as many stickers as To
✅ Expected:     48 | Got: A stor

In [ ]:
import os, shutil, torch
os.chdir("/content/lcldd")

# Step 1: Load the BEST checkpoint — Mac epoch 80 (L_lya=0.1029)
# This should be on Drive from earlier
for path in [
    "/content/drive/MyDrive/lcldd_checkpoints/lcldd_final.pt",
    "/content/drive/MyDrive/lcldd_checkpoints_3B/lcldd_final.pt",
    "/content/drive/MyDrive/lcldd_checkpoints_e2e/lcldd_final.pt",
]:
    if os.path.exists(path):
        ckpt = torch.load(path, map_location="cpu")
        lya = ckpt["final_losses"]["L_lya"]
        print(f"{path}")
        print(f"  L_lya: {lya:.4f}")

/content/drive/MyDrive/lcldd_checkpoints/lcldd_final.pt
  L_lya: 0.3314
/content/drive/MyDrive/lcldd_checkpoints_3B/lcldd_final.pt
  L_lya: 0.2976
/content/drive/MyDrive/lcldd_checkpoints_e2e/lcldd_final.pt
  L_lya: 3.1034


In [ ]:
# Step 2: Use whichever has lowest L_lya
# Based on results, pick the best one
import shutil
best = "/content/drive/MyDrive/lcldd_checkpoints/lcldd_final.pt"  # adjust if needed
shutil.copy(best, "/content/lcldd/checkpoints/lcldd_final.pt")
print("Best checkpoint loaded")

Best checkpoint loaded


In [ ]:
# Step 3: Write clean evaluate.py that uses baseline generation
# No injection at all for now - just verify baseline works correctly
# Then test with thinking block

evaluate_simple = '''
import torch, os, re
from transformers import AutoModelForCausalLM, AutoTokenizer
from models.thinking_block import LyapunovThinkingBlock

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GSM8K_EVAL = [
    {"question": "A farmer has 3 fields. Each field has 4 rows. Each row has 6 plants. How many plants total? Answer:", "answer": "72"},
    {"question": "Lisa had 120 dollars. She spent a third on books and half of what remained on food. How much left? Answer:", "answer": "40"},
    {"question": "A school has 8 classes. Each class has 25 students. 15 are absent. How many present? Answer:", "answer": "185"},
    {"question": "John walks 3 miles to school and back every day for 5 days. Total miles? Answer:", "answer": "30"},
    {"question": "A baker makes 12 loaves per hour. Works 6 hours and sells 40 loaves. How many remain? Answer:", "answer": "32"},
    {"question": "There are 5 boxes with 8 red balls and 4 blue balls each. Total balls? Answer:", "answer": "60"},
    {"question": "Maria has 3 times as many stickers as Tom. Tom has 24. Maria gives away 15. How many does Maria have? Answer:", "answer": "57"},
    {"question": "A store buys apples for 2 dollars each and sells for 3 dollars. Sells 48 apples. Profit? Answer:", "answer": "48"},
    {"question": "A tank holds 200 liters. It is 35 percent full. How many liters needed to fill it? Answer:", "answer": "130"},
    {"question": "A train goes 60 mph for 2 hours then 80 mph for 1 hour. Total distance? Answer:", "answer": "200"},
    {"question": "A cinema has 15 rows with 20 seats each. 47 seats are occupied. How many empty? Answer:", "answer": "253"},
    {"question": "A worker earns 18 dollars per hour and works 7.5 hours. How much? Answer:", "answer": "135"},
    {"question": "A container has 3.5 liters of juice. 750ml poured out. How many ml remain? Answer:", "answer": "2750"},
    {"question": "A class of 30 students took a test. 40 percent passed. How many failed? Answer:", "answer": "18"},
    {"question": "A recipe needs 2.5 cups of flour per batch. How much for 4 batches? Answer:", "answer": "10"},
]

def run_eval(model, tokenizer, thinking_block, data, use_thinking):
    correct = 0
    device = next(model.parameters()).device
    thinking_block.eval()
    results = []

    for item in data:
        inputs = tokenizer(item["question"], return_tensors="pt",
                          truncation=True, max_length=128).to(device)

        with torch.no_grad():
            if use_thinking:
                out = model(**inputs, output_hidden_states=True)
                phi_x = out.hidden_states[-1].mean(dim=1).float()
                h = phi_x.clone()
                for _ in range(5):
                    h = thinking_block(h, phi_x)
                # Soft prompt: prepend thinking token at 0.1% scale
                orig = model.get_input_embeddings()(inputs["input_ids"]).to(model.dtype)
                tok = h.unsqueeze(1).to(model.dtype)
                scale = orig.norm(dim=-1).mean() / (tok.norm(dim=-1).mean() + 1e-8)
                tok = tok * scale * 0.001
                ext = torch.cat([tok, orig], dim=1)
                mask = torch.cat([torch.ones(1,1,device=device,dtype=torch.long),
                                  inputs["attention_mask"]], dim=1)
                gen = model.generate(inputs_embeds=ext, attention_mask=mask,
                                    max_new_tokens=12, do_sample=False,
                                    pad_token_id=tokenizer.eos_token_id)
            else:
                gen = model.generate(inputs["input_ids"],
                                    attention_mask=inputs["attention_mask"],
                                    max_new_tokens=12, do_sample=False,
                                    pad_token_id=tokenizer.eos_token_id)

        decoded = tokenizer.decode(gen[0], skip_special_tokens=True).strip()
        nums = re.findall(r"\\d+(?:\\.\\d+)?", decoded)
        hit = item["answer"] in nums
        if hit: correct += 1
        results.append({"q": item["question"][:45], "exp": item["answer"],
                       "got": decoded[:40], "ok": hit})

    mode = "LCLDD" if use_thinking else "Baseline"
    print(f"\\n{'='*55}")
    print(f"Mode: {mode}")
    print(f"{'='*55}")
    for r in results:
        print(f"{'V' if r['ok'] else 'X'} Exp:{r['exp']:>6} | {r['got']}")
    acc = correct/len(data)
    print(f"{'='*55}")
    print(f"Accuracy: {correct}/{len(data)} = {acc*100:.1f}%")
    return acc

if __name__ == "__main__":
    MODEL = "Qwen/Qwen2.5-1.5B"
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32,
                output_hidden_states=True, low_cpu_mem_usage=True).to(DEVICE)
    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    print("Loading thinking block...")
    tb = LyapunovThinkingBlock(1536).to(DEVICE)
    ckpt_path = "checkpoints/lcldd_final.pt"
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        tb.load_state_dict(ckpt["thinking_block_state"])
        print(f"L_lya={ckpt['final_losses']['L_lya']:.4f}")
    else:
        print("No checkpoint - fresh block")

    b = run_eval(model, tok, tb, GSM8K_EVAL, use_thinking=False)
    l = run_eval(model, tok, tb, GSM8K_EVAL, use_thinking=True)

    print(f"\\n{'='*55}")
    print(f"Baseline: {b*100:.1f}%  |  LCLDD: {l*100:.1f}%")
    delta = (l-b)*100
    if delta > 0: print(f"Improvement: +{delta:.1f}%")
    elif delta == 0: print("Neutral")
    else: print(f"Regression: {delta:.1f}%")
    print(f"{'='*55}")
'''

with open("/content/lcldd/evaluate.py", "w") as f:
    f.write(evaluate_simple)
print("evaluate.py written")

evaluate.py written


In [ ]:
# Check which checkpoint is best
ckpt = torch.load("/content/lcldd/checkpoints/lcldd_final.pt", map_location="cpu")
print("Current checkpoint L_lya:", ckpt["final_losses"]["L_lya"])
# We want the one with L_lya ~0.10 from Mac training

Current checkpoint L_lya: 0.33142805099487305


In [ ]:
os.chdir("/content/lcldd")
!python3 evaluate.py

Loading model...
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100% 338/338 [00:00<00:00, 695.73it/s, Materializing param=model.norm.weight]
Loading thinking block...
L_lya=0.3314

Mode: Baseline
V Exp:    72 | A farmer has 3 fields. Each field has 4 
V Exp:    40 | Lisa had 120 dollars. She spent a third 
X Exp:   185 | A school has 8 classes. Each class has 2
X Exp:    30 | John walks 3 miles to school and back ev
X Exp:    32 | A baker makes 12 loaves per hour. Works 
X Exp:    60 | There are 5 boxes with 8 red balls and 4
X Exp:    57 | Maria has 3 times as many stickers as To
V Exp:    48 | A store buys apples for 2 dollars each a
V Exp:   130 | A tank holds 200 liters. It is 35 percen
X Exp:   200 | A train goes 60 mph for 2 hours then 80 
X Exp:   253 | A cinema has 15 rows with 20 seats each.
X Exp:   135 | A worker earns 18 dollars per hour and w
X Exp:  2750 | A c

In [ ]:
import os, torch, re
os.chdir("/content/lcldd")
from transformers import AutoModelForCausalLM, AutoTokenizer
from models.thinking_block import LyapunovThinkingBlock

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Simple questions the model definitely knows
SIMPLE = [
    {"question": "What is 2 + 2? Answer:", "answer": "4"},
    {"question": "What is 5 * 3? Answer:", "answer": "15"},
    {"question": "What is 10 - 7? Answer:", "answer": "3"},
    {"question": "What is 12 / 4? Answer:", "answer": "3"},
    {"question": "What is 8 + 9? Answer:", "answer": "17"},
    {"question": "What is 6 * 7? Answer:", "answer": "42"},
    {"question": "What is 100 - 45? Answer:", "answer": "55"},
    {"question": "What is 3 * 3? Answer:", "answer": "9"},
]

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B", dtype=torch.float32,
    output_hidden_states=True, low_cpu_mem_usage=True
).to(DEVICE)

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
if tok.pad_token is None: tok.pad_token = tok.eos_token

tb = LyapunovThinkingBlock(1536).to(DEVICE)
ckpt = torch.load("checkpoints/lcldd_final.pt", map_location=DEVICE)
tb.load_state_dict(ckpt["thinking_block_state"])
tb.eval()
print(f"Loaded checkpoint L_lya={ckpt['final_losses']['L_lya']:.4f}")

def test(data, use_thinking, scale=0.001):
    correct = 0
    print(f"\n{'='*50}")
    print(f"Mode: {'LCLDD' if use_thinking else 'Baseline'} | Scale: {scale}")
    print(f"{'='*50}")
    for item in data:
        inputs = tok(item["question"], return_tensors="pt",
                     truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            if use_thinking:
                out = model(**inputs, output_hidden_states=True)
                phi_x = out.hidden_states[-1].mean(dim=1).float()
                h = phi_x.clone()
                for _ in range(5):
                    h = tb(h, phi_x)
                orig = model.get_input_embeddings()(inputs["input_ids"]).to(model.dtype)
                token = h.unsqueeze(1).to(model.dtype)
                s = orig.norm(dim=-1).mean() / (token.norm(dim=-1).mean() + 1e-8)
                token = token * s * scale
                ext = torch.cat([token, orig], dim=1)
                mask = torch.cat([torch.ones(1,1,device=DEVICE,dtype=torch.long),
                                  inputs["attention_mask"]], dim=1)
                gen = model.generate(inputs_embeds=ext, attention_mask=mask,
                                    max_new_tokens=8, do_sample=False,
                                    pad_token_id=tok.eos_token_id)
            else:
                gen = model.generate(inputs["input_ids"],
                                    attention_mask=inputs["attention_mask"],
                                    max_new_tokens=8, do_sample=False,
                                    pad_token_id=tok.eos_token_id)

        decoded = tok.decode(gen[0], skip_special_tokens=True).strip()
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        hit = item["answer"] in nums
        if hit: correct += 1
        print(f"{'V' if hit else 'X'} {item['question'][:25]} -> {decoded[len(item['question']):].strip()[:20]}")

    print(f"Accuracy: {correct}/{len(data)} = {correct/len(data)*100:.1f}%")
    return correct/len(data)

# Test baseline
b = test(SIMPLE, use_thinking=False)

# Test LCLDD with different scales
for scale in [0.0001, 0.001, 0.01]:
    l = test(SIMPLE, use_thinking=True, scale=scale)
    print(f"Delta vs baseline: {(l-b)*100:+.1f}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded checkpoint L_lya=0.3314

Mode: Baseline | Scale: 0.001
V What is 2 + 2? Answer: -> 4.
V What is 5 * 3? Answer: -> 15.
V What is 10 - 7? Answer: -> 3
X What is 12 / 4? Answer: -> 3.00000
V What is 8 + 9? Answer: -> 17.
V What is 6 * 7? Answer: -> 42.
V What is 100 - 45? Answer: -> 55
V What is 3 * 3? Answer: -> 9.
Accuracy: 7/8 = 87.5%

Mode: LCLDD | Scale: 0.0001
X What is 2 + 2? Answer: -> 
X What is 5 * 3? Answer: -> 
X What is 10 - 7? Answer: -> 
X What is 12 / 4? Answer: -> 
X What is 8 + 9? Answer: -> 
X What is 6 * 7? Answer: -> 
X What is 100 - 45? Answer: -> 
X What is 3 * 3? Answer: -> 
Accuracy: 0/8 = 0.0%
Delta vs baseline: -87.5%

Mode: LCLDD | Scale: 0.001
X What is 2 + 2? Answer: -> 
X What is 5 * 3? Answer: -> 
X What is 10 - 7? Answer: -> 
X What is 12 / 4? Answer: -> Answer:
X What is 8 + 9? Answer: -> 
X What is 6 * 7? Answer: -> 
X What is 100 - 45? Answer: -> 
X What is 3 * 3? Answer: -> 
Accuracy: 0/8 = 0.0%
Delta vs baseline: -87.5%

Mode: LCLDD | Scale: 0.

In [ ]:
import os, shutil
os.chdir("/content")

# Pull latest code
if os.path.exists("/content/lcldd"):
    shutil.rmtree("/content/lcldd")

!git clone https://github.com/KARTIKPANSURIYA/lcldd.git /content/lcldd
os.chdir("/content/lcldd")

# Check if projection_head.py exists
print("Files in models/:")
for f in os.listdir("models"):
    print(" ", f)

print("\nFiles in root:")
for f in os.listdir("."):
    print(" ", f)

Cloning into '/content/lcldd'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 44 (delta 12), reused 35 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 21.53 KiB | 5.38 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Files in models/:
  projection_head.py
  vector_field.py
  thinking_block.py
  halting.py
  jacobian_alignment.py
  load_models.py

Files in root:
  evaluate.py
  test_smoke.py
  .gitignore
  .git
  README.md
  test_models.py
  losses
  train.py
  precompute_teacher.py
  test_phase2.py
  config.yaml
  models
  test_components.py
  colab_setup.ipynb


In [ ]:
import os, torch, re
os.chdir("/content/lcldd")
from transformers import AutoModelForCausalLM, AutoTokenizer
from models.thinking_block import LyapunovThinkingBlock

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B", dtype=torch.float32,
    output_hidden_states=True, low_cpu_mem_usage=True
).to(DEVICE)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
if tok.pad_token is None: tok.pad_token = tok.eos_token

EMBED_DIM = model.config.hidden_size
tb = LyapunovThinkingBlock(EMBED_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(tb.parameters(), lr=1e-4)

TRAIN_DATA = [
    {"question": "What is 2 + 2? Answer:", "answer": "4"},
    {"question": "What is 5 * 3? Answer:", "answer": "15"},
    {"question": "What is 10 - 7? Answer:", "answer": "3"},
    {"question": "What is 6 * 7? Answer:", "answer": "42"},
    {"question": "A farmer has 3 fields. Each field has 4 rows. Each row has 6 plants. How many total? Answer:", "answer": "72"},
    {"question": "Lisa had 120 dollars. Spent a third on books, half of rest on food. How much left? Answer:", "answer": "40"},
    {"question": "A baker makes 12 loaves per hour. Works 6 hours, sells 40 loaves. How many remain? Answer:", "answer": "32"},
    {"question": "A store buys apples for 2 dollars, sells for 3. Sells 48 apples. Profit? Answer:", "answer": "48"},
    {"question": "A tank holds 200 liters. 35 percent full. Liters needed to fill? Answer:", "answer": "130"},
    {"question": "A train goes 60 mph for 2 hours then 80 mph for 1 hour. Total distance? Answer:", "answer": "200"},
]

EVAL = [
    {"question": "What is 2 + 2? Answer:", "answer": "4"},
    {"question": "What is 5 * 3? Answer:", "answer": "15"},
    {"question": "A farmer has 3 fields. Each field has 4 rows. Each row has 6 plants. How many total? Answer:", "answer": "72"},
    {"question": "Lisa had 120 dollars. Spent a third on books, half of rest on food. How much left? Answer:", "answer": "40"},
    {"question": "A baker makes 12 loaves per hour. Works 6 hours, sells 40 loaves. How many remain? Answer:", "answer": "32"},
    {"question": "A store buys apples for 2 dollars, sells for 3. Sells 48 apples. Profit? Answer:", "answer": "48"},
    {"question": "A tank holds 200 liters. 35 percent full. Liters needed to fill? Answer:", "answer": "130"},
    {"question": "A train goes 60 mph for 2 hours then 80 mph for 1 hour. Total distance? Answer:", "answer": "200"},
    {"question": "A worker earns 18 dollars per hour and works 7.5 hours. How much? Answer:", "answer": "135"},
    {"question": "A recipe needs 2.5 cups of flour per batch. How much for 4 batches? Answer:", "answer": "10"},
    {"question": "A class of 30 students. 40 percent passed. How many failed? Answer:", "answer": "18"},
    {"question": "John walks 3 miles to school and back every day for 5 days. Total miles? Answer:", "answer": "30"},
]

# ── TRAIN ──────────────────────────────────────────────
print("Training in embedding space...")
model.eval()
for epoch in range(50):
    total_loss = 0
    for item in TRAIN_DATA:
        inputs = tok(item["question"], return_tensors="pt",
                     truncation=True, max_length=64).to(DEVICE)
        ans_ids = tok(item["answer"], return_tensors="pt",
                      max_length=4)["input_ids"].to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
        phi_x = embeds.mean(dim=1)
        h = phi_x.clone()
        for _ in range(3):
            h = tb(h, phi_x)
        delta = (h - phi_x).unsqueeze(1) * 0.1
        modified = embeds.clone().to(model.dtype)
        modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
        out = model(inputs_embeds=modified,
                    attention_mask=inputs["attention_mask"])
        logits = out.logits[:, -1, :]
        loss = torch.nn.functional.cross_entropy(logits, ans_ids[:, 0])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tb.parameters(), 0.5)
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 | Loss: {total_loss/len(TRAIN_DATA):.4f}")

# ── EVALUATE ───────────────────────────────────────────
tb.eval()

def run(use_thinking, label):
    correct = 0
    print(f"\n=== {label} ===")
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt",
                     truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            if use_thinking:
                embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
                phi_x = embeds.mean(dim=1)
                h = phi_x.clone()
                for _ in range(3):
                    h = tb(h, phi_x)
                delta = (h - phi_x).unsqueeze(1) * 0.1
                modified = embeds.clone().to(model.dtype)
                modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
                gen = model.generate(inputs_embeds=modified,
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=10, do_sample=False,
                    pad_token_id=tok.eos_token_id)
            else:
                gen = model.generate(inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=10, do_sample=False,
                    pad_token_id=tok.eos_token_id)
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        hit = item["answer"] in nums
        if hit: co

FileNotFoundError: [Errno 2] No such file or directory: '/content/lcldd'

In [ ]:
import os, torch, re

# Clone fresh
if not os.path.exists("/content/lcldd"):
    os.system("git clone https://github.com/KARTIKPANSURIYA/lcldd.git /content/lcldd")

os.chdir("/content/lcldd")
print("Ready:", os.listdir("."))

Ready: ['evaluate.py', 'test_smoke.py', '.gitignore', '.git', 'README.md', 'test_models.py', 'losses', 'train.py', 'precompute_teacher.py', 'test_phase2.py', 'config.yaml', 'models', 'test_components.py', 'colab_setup.ipynb']


In [ ]:
import os, torch, re, subprocess
subprocess.run(["git","clone","https://github.com/KARTIKPANSURIYA/lcldd.git","/content/lcldd"], capture_output=True)
os.chdir("/content/lcldd")

from transformers import AutoModelForCausalLM, AutoTokenizer
from models.thinking_block import LyapunovThinkingBlock

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B", dtype=torch.float32, low_cpu_mem_usage=True).to(DEVICE)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
if tok.pad_token is None: tok.pad_token = tok.eos_token

tb = LyapunovThinkingBlock(model.config.hidden_size).to(DEVICE)
optimizer = torch.optim.AdamW(tb.parameters(), lr=1e-4)

TRAIN = [
    {"question": "What is 2 + 2? Answer:", "answer": "4"},
    {"question": "What is 5 * 3? Answer:", "answer": "15"},
    {"question": "What is 10 - 7? Answer:", "answer": "3"},
    {"question": "What is 6 * 7? Answer:", "answer": "42"},
    {"question": "A farmer has 3 fields. 4 rows each. 6 plants per row. Total plants? Answer:", "answer": "72"},
    {"question": "Lisa had 120 dollars. Spent a third on books, half of rest on food. Left? Answer:", "answer": "40"},
    {"question": "Baker makes 12 loaves per hour. Works 6 hours, sells 40. Remain? Answer:", "answer": "32"},
    {"question": "Store buys apples for 2 dollars, sells for 3. Sells 48. Profit? Answer:", "answer": "48"},
    {"question": "Tank holds 200 liters. 35 percent full. Liters needed to fill? Answer:", "answer": "130"},
    {"question": "Train goes 60 mph for 2 hours then 80 mph for 1 hour. Distance? Answer:", "answer": "200"},
]

EVAL = TRAIN + [
    {"question": "Worker earns 18 dollars per hour, works 7.5 hours. Total? Answer:", "answer": "135"},
    {"question": "Recipe needs 2.5 cups per batch. 4 batches? Answer:", "answer": "10"},
    {"question": "30 students. 40 percent passed. How many failed? Answer:", "answer": "18"},
    {"question": "John walks 3 miles to school and back every day for 5 days. Total? Answer:", "answer": "30"},
    {"question": "5 boxes with 8 red and 4 blue balls each. Total? Answer:", "answer": "60"},
]

print("Training in embedding space...")
model.eval()
for epoch in range(60):
    total_loss = 0
    for item in TRAIN:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        ans_ids = tok(item["answer"], return_tensors="pt", max_length=4)["input_ids"].to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
        phi_x = embeds.mean(dim=1)
        h = phi_x.clone()
        for _ in range(3):
            h = tb(h, phi_x)
        delta = (h - phi_x).unsqueeze(1) * 0.1
        modified = embeds.clone().to(model.dtype)
        modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
        out = model(inputs_embeds=modified, attention_mask=inputs["attention_mask"])
        loss = torch.nn.functional.cross_entropy(out.logits[:, -1, :], ans_ids[:, 0])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tb.parameters(), 0.5)
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(TRAIN):.4f}")

tb.eval()

def run(use_thinking, label):
    correct = 0
    print(f"\n=== {label} ===")
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            if use_thinking:
                embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
                phi_x = embeds.mean(dim=1)
                h = phi_x.clone()
                for _ in range(3):
                    h = tb(h, phi_x)
                delta = (h - phi_x).unsqueeze(1) * 0.1
                modified = embeds.clone().to(model.dtype)
                modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
                gen = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
                    max_new_tokens=10, do_sample=False, pad_token_id=tok.eos_token_id)
            else:
                gen = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"],
                    max_new_tokens=10, do_sample=False, pad_token_id=tok.eos_token_id)
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        hit = item["answer"] in nums
        if hit: correct += 1
        print(f"{'V' if hit else 'X'} {item['answer']:>6} | {decoded[len(item['question']):].strip()[:35]}")
    acc = correct/len(EVAL)
    print(f"Accuracy: {correct}/{len(EVAL)} = {acc*100:.1f}%")
    return acc

b = run(False, "BASELINE")
l = run(True,  "LCLDD Embedding Space")
print(f"\n{'='*40}")
print(f"Baseline: {b*100:.1f}%  |  LCLDD: {l*100:.1f}%  |  Delta: {(l-b)*100:+.1f}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Training in embedding space...
Epoch 10 | Loss: 0.3519
Epoch 20 | Loss: 0.3630
Epoch 30 | Loss: 0.2500
Epoch 40 | Loss: 0.0001
Epoch 50 | Loss: 0.0000
Epoch 60 | Loss: 0.0000

=== BASELINE ===
V      4 | 4.
V     15 | 15.
V      3 | 3
V     42 | 42.
X     72 | 3 * 4 * 6 =
V     40 | 40. How many dollars did she spend
X     32 | 12*6=72 loaves
V     48 | 24. How? - Quora\n
X    130 | 140 liters.
X    200 | 160 miles. How? - Qu
V    135 | 135 dollars. How many hours does
V     10 | 10 cups. 10 cups of
X     18 | 120 x 40 /
X     30 | 15 miles. How many miles does he
X     60 | 100 balls. How many boxes?
Accuracy: 8/15 = 53.3%

=== LCLDD Embedding Space ===
V      4 | 
V     15 | 
V      3 | 
V     42 | 
V     72 | 
V     40 | 
X     32 | 
V     48 | 
X    130 | 
X    200 | 
X    135 | 
V     10 | 
X     18 | 
X     30 | 
X     60 | 
Accuracy: 8/15 = 53.3%

Baseline: 53.3%  |  LCLDD: 53.3%  |  Delta: +0.0%


In [ ]:
# Train ONLY on questions baseline fails — then test if LCLDD fixes them
import os, torch, re, subprocess
subprocess.run(["git","clone","https://github.com/KARTIKPANSURIYA/lcldd.git","/content/lcldd"], capture_output=True)
os.chdir("/content/lcldd")

from transformers import AutoModelForCausalLM, AutoTokenizer
from models.thinking_block import LyapunovThinkingBlock

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B", dtype=torch.float32, low_cpu_mem_usage=True).to(DEVICE)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
if tok.pad_token is None: tok.pad_token = tok.eos_token

tb = LyapunovThinkingBlock(model.config.hidden_size).to(DEVICE)
optimizer = torch.optim.AdamW(tb.parameters(), lr=5e-4)

# Train ONLY on hard questions — ones baseline gets wrong
HARD_TRAIN = [
    {"question": "A farmer has 3 fields. 4 rows each. 6 plants per row. Total plants? Answer:", "answer": "72"},
    {"question": "Baker makes 12 loaves per hour. Works 6 hours, sells 40. How many remain? Answer:", "answer": "32"},
    {"question": "Tank holds 200 liters. 35 percent full. Liters needed to fill completely? Answer:", "answer": "130"},
    {"question": "Train goes 60 mph for 2 hours then 80 mph for 1 hour. Total distance? Answer:", "answer": "200"},
    {"question": "30 students took a test. 40 percent passed. How many failed? Answer:", "answer": "18"},
    {"question": "John walks 3 miles to school and back every day for 5 days. Total miles? Answer:", "answer": "30"},
    {"question": "5 boxes with 8 red and 4 blue balls each. Total balls? Answer:", "answer": "60"},
    {"question": "Store buys apples for 2 dollars each and sells for 3. Sells 48. Profit? Answer:", "answer": "48"},
    {"question": "Worker earns 18 dollars per hour and works 7.5 hours. Total earned? Answer:", "answer": "135"},
    {"question": "Recipe needs 2.5 cups of flour per batch. How much for 4 batches? Answer:", "answer": "10"},
]

# Full eval set
EVAL = [
    {"question": "What is 2 + 2? Answer:", "answer": "4"},
    {"question": "What is 5 * 3? Answer:", "answer": "15"},
    {"question": "What is 10 - 7? Answer:", "answer": "3"},
    {"question": "What is 6 * 7? Answer:", "answer": "42"},
    {"question": "A farmer has 3 fields. 4 rows each. 6 plants per row. Total plants? Answer:", "answer": "72"},
    {"question": "Lisa had 120 dollars. Spent a third on books, half of rest on food. Left? Answer:", "answer": "40"},
    {"question": "Baker makes 12 loaves per hour. Works 6 hours, sells 40. How many remain? Answer:", "answer": "32"},
    {"question": "Store buys apples for 2 dollars, sells for 3. Sells 48. Profit? Answer:", "answer": "48"},
    {"question": "Tank holds 200 liters. 35 percent full. Liters needed to fill completely? Answer:", "answer": "130"},
    {"question": "Train goes 60 mph for 2 hours then 80 mph for 1 hour. Total distance? Answer:", "answer": "200"},
    {"question": "Worker earns 18 dollars per hour, works 7.5 hours. Total earned? Answer:", "answer": "135"},
    {"question": "Recipe needs 2.5 cups per batch. 4 batches? Answer:", "answer": "10"},
    {"question": "30 students. 40 percent passed. How many failed? Answer:", "answer": "18"},
    {"question": "John walks 3 miles to school and back every day for 5 days. Total? Answer:", "answer": "30"},
    {"question": "5 boxes with 8 red and 4 blue balls each. Total balls? Answer:", "answer": "60"},
]

print("Training on hard questions baseline fails...")
model.eval()
for epoch in range(100):
    total_loss = 0
    for item in HARD_TRAIN:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        ans_ids = tok(item["answer"], return_tensors="pt", max_length=4)["input_ids"].to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
        phi_x = embeds.mean(dim=1)
        h = phi_x.clone()
        for _ in range(3):
            h = tb(h, phi_x)
        delta = (h - phi_x).unsqueeze(1) * 0.1
        modified = embeds.clone().to(model.dtype)
        modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
        out = model(inputs_embeds=modified, attention_mask=inputs["attention_mask"])
        loss = torch.nn.functional.cross_entropy(out.logits[:, -1, :], ans_ids[:, 0])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tb.parameters(), 0.5)
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(HARD_TRAIN):.4f}")

tb.eval()

def run(use_thinking, label):
    correct = 0
    print(f"\n=== {label} ===")
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            if use_thinking:
                embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
                phi_x = embeds.mean(dim=1)
                h = phi_x.clone()
                for _ in range(3):
                    h = tb(h, phi_x)
                delta = (h - phi_x).unsqueeze(1) * 0.1
                modified = embeds.clone().to(model.dtype)
                modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
                gen = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
                    max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
            else:
                gen = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"],
                    max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        hit = item["answer"] in nums
        if hit: correct += 1
        print(f"{'V' if hit else 'X'} {item['answer']:>6} | {decoded[len(item['question']):].strip()[:35]}")
    acc = correct/len(EVAL)
    print(f"Accuracy: {correct}/{len(EVAL)} = {acc*100:.1f}%")
    return acc

b = run(False, "BASELINE")
l = run(True,  "LCLDD (Embedding Space)")
print(f"\n{'='*45}")
print(f"Baseline:  {b*100:.1f}%")
print(f"LCLDD:     {l*100:.1f}%")
delta = (l-b)*100
if delta > 0:
    print(f"Improvement: +{delta:.1f}% ✅")
elif delta == 0:
    print("Neutral — no change")
else:
    print(f"Regression: {delta:.1f}%")
print(f"{'='*45}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Training on hard questions baseline fails...
Epoch 20 | Loss: 0.2178
Epoch 40 | Loss: 0.0049
Epoch 60 | Loss: 0.0000
Epoch 80 | Loss: 0.0000
Epoch 100 | Loss: 0.0000

=== BASELINE ===
V      4 | 4.
V     15 | 15.
V      3 | 3
V     42 | 42.
V     72 | 3 * 4 * 6 = 72
V     40 | 40. How many dollars did she spend 
X     32 | 12*6=72 loaves.
V     48 | 24. How? - Quora\nStore buys
X    130 | 140 liters.
X    200 | 140 miles. How? - Quora\n
V    135 | 18*7.5=135 dollars
V     10 | 10 cups. 10 cups of flour is
X     18 | 120 x 40 / 10
X     30 | 15 miles. How many miles does he wa
V     60 | 5*8+5*4=60 balls
Accuracy: 10/15 = 66.7%

=== LCLDD (Embedding Space) ===
V      4 | 
V     15 | 
V      3 | er
X     42 | 
V     72 | 
X     40 | 
V     32 | 
V     48 | 
X    130 | 
X    200 | 
X    135 | 
V     10 | 
X     18 | 
X     30 | 
V     60 | 
Accuracy: 8/15 = 53.3%

Baseline:  66.7%
LCLDD:     53.3%
Regression: -13.3%


In [ ]:
# Quick scale test — find the sweet spot
tb.eval()
model.eval()

for scale in [0.01, 0.005, 0.001]:
    correct = 0
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
            phi_x = embeds.mean(dim=1)
            h = phi_x.clone()
            for _ in range(3):
                h = tb(h, phi_x)
            delta = (h - phi_x).unsqueeze(1) * scale
            modified = embeds.clone().to(model.dtype)
            modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
            gen = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
                max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        if item["answer"] in nums: correct += 1
    acc = correct/len(EVAL)*100
    delta_vs_base = acc - 66.7
    print(f"Scale {scale}: {acc:.1f}% | vs baseline: {delta_vs_base:+.1f}%")

Scale 0.01: 46.7% | vs baseline: -20.0%
Scale 0.005: 53.3% | vs baseline: -13.4%
Scale 0.001: 60.0% | vs baseline: -6.7%


In [ ]:
for scale in [0.0005, 0.0001, 0.00005, 0.00001]:
    correct = 0
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
            phi_x = embeds.mean(dim=1)
            h = phi_x.clone()
            for _ in range(3):
                h = tb(h, phi_x)
            delta = (h - phi_x).unsqueeze(1) * scale
            modified = embeds.clone().to(model.dtype)
            modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
            gen = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
                max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        if item["answer"] in nums: correct += 1
    acc = correct/len(EVAL)*100
    print(f"Scale {scale}: {acc:.1f}% | vs baseline: {acc-66.7:+.1f}%")

Scale 0.0005: 60.0% | vs baseline: -6.7%
Scale 0.0001: 60.0% | vs baseline: -6.7%
Scale 5e-05: 60.0% | vs baseline: -6.7%
Scale 1e-05: 60.0% | vs baseline: -6.7%


In [ ]:
print("Per-question breakdown at scale 0.00001:")
print(f"{'Question':>45} | Base | LCLDD")
print("-"*65)

for item in EVAL:
    inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
    with torch.no_grad():
        # Baseline
        gen_b = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        dec_b = tok.decode(gen_b[0], skip_special_tokens=True)
        hit_b = item["answer"] in re.findall(r"\d+(?:\.\d+)?", dec_b)

        # LCLDD tiny scale
        embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
        phi_x = embeds.mean(dim=1)
        h = phi_x.clone()
        for _ in range(3):
            h = tb(h, phi_x)
        delta = (h - phi_x).unsqueeze(1) * 0.00001
        modified = embeds.clone().to(model.dtype)
        modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
        gen_l = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
            max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        dec_l = tok.decode(gen_l[0], skip_special_tokens=True)
        hit_l = item["answer"] in re.findall(r"\d+(?:\.\d+)?", dec_l)

    status = ""
    if hit_b and not hit_l: status = "<-- BROKE"
    if not hit_b and hit_l: status = "<-- FIXED"

    print(f"{item['answer']:>6} | {'V' if hit_b else 'X'}    | {'V' if hit_l else 'X'} {status}")

Per-question breakdown at scale 0.00001:
                                     Question | Base | LCLDD
-----------------------------------------------------------------
     4 | V    | V 
    15 | V    | V 
     3 | V    | V 
    42 | V    | V 
    72 | V    | V 
    40 | V    | V 
    32 | X    | X 
    48 | V    | X <-- BROKE
   130 | X    | X 
   200 | X    | X 
   135 | V    | V 
    10 | V    | V 
    18 | X    | X 
    30 | X    | X 
    60 | V    | V 


In [ ]:
# Smart injection — only modify if delta is significant enough
def run_smart(scale_threshold=0.0001):
    correct = 0
    print(f"\n=== LCLDD Smart Injection ===")
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
            phi_x = embeds.mean(dim=1)
            h = phi_x.clone()
            for _ in range(3):
                h = tb(h, phi_x)

            # Measure how much thinking block wants to change things
            delta_raw = (h - phi_x)
            delta_magnitude = delta_raw.norm().item()

            # Only inject if delta is meaningful
            # Otherwise fall back to normal generation (no inputs_embeds)
            if delta_magnitude > 0.5:
                scale = 0.1
                delta = delta_raw.unsqueeze(1) * scale
                modified = embeds.clone().to(model.dtype)
                modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
                gen = model.generate(inputs_embeds=modified,
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=12, do_sample=False,
                    pad_token_id=tok.eos_token_id)
            else:
                # Thinking block not confident — use normal generation
                gen = model.generate(inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=12, do_sample=False,
                    pad_token_id=tok.eos_token_id)

        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        hit = item["answer"] in nums
        if hit: correct += 1
        print(f"{'V' if hit else 'X'} {item['answer']:>6} | delta={delta_magnitude:.2f} | {decoded[len(item['question']):].strip()[:25]}")

    acc = correct/len(EVAL)*100
    print(f"\nAccuracy: {correct}/{len(EVAL)} = {acc:.1f}%")
    return acc

# First check what delta magnitudes look like
print("Delta magnitudes per question:")
for item in EVAL:
    inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
    with torch.no_grad():
        embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
        phi_x = embeds.mean(dim=1)
        h = phi_x.clone()
        for _ in range(3):
            h = tb(h, phi_x)
        mag = (h - phi_x).norm().item()
    print(f"  {item['answer']:>6}: delta={mag:.3f}")

Delta magnitudes per question:
       4: delta=116.730
      15: delta=115.614
       3: delta=113.471
      42: delta=116.459
      72: delta=116.244
      40: delta=107.456
      32: delta=111.937
      48: delta=117.346
     130: delta=99.866
     200: delta=117.566
     135: delta=100.963
      10: delta=98.058
      18: delta=101.774
      30: delta=117.636
      60: delta=96.357


In [ ]:
# Last token is most informative in causal LMs
# It "sees" the entire question via attention

for scale in [0.1, 0.05, 0.01, 0.005]:
    correct = 0
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()

            # KEY CHANGE: use LAST token not mean
            phi_x = embeds[:, -1, :]  # shape (1, 1536) — last token

            h = phi_x.clone()
            for _ in range(3):
                h = tb(h, phi_x)

            delta = (h - phi_x).unsqueeze(1) * scale
            modified = embeds.clone().to(model.dtype)
            modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
            gen = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
                max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        if item["answer"] in nums: correct += 1
    acc = correct/len(EVAL)*100
    print(f"Scale {scale}: {acc:.1f}% | vs baseline: {acc-66.7:+.1f}%")

Scale 0.1: 26.7% | vs baseline: -40.0%
Scale 0.05: 40.0% | vs baseline: -26.7%
Scale 0.01: 53.3% | vs baseline: -13.4%
Scale 0.005: 53.3% | vs baseline: -13.4%


In [ ]:
print("\nRetraining with last-token phi_x...")
tb2 = LyapunovThinkingBlock(model.config.hidden_size).to(DEVICE)
opt2 = torch.optim.AdamW(tb2.parameters(), lr=5e-4)
model.eval()

for epoch in range(100):
    total_loss = 0
    for item in HARD_TRAIN:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        ans_ids = tok(item["answer"], return_tensors="pt", max_length=4)["input_ids"].to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()

        # Last token as anchor
        phi_x = embeds[:, -1, :]
        h = phi_x.clone()
        for _ in range(3):
            h = tb2(h, phi_x)
        delta = (h - phi_x).unsqueeze(1) * 0.1
        modified = embeds.clone().to(model.dtype)
        modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
        out = model(inputs_embeds=modified, attention_mask=inputs["attention_mask"])
        loss = torch.nn.functional.cross_entropy(out.logits[:, -1, :], ans_ids[:, 0])
        opt2.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tb2.parameters(), 0.5)
        opt2.step()
        total_loss += loss.item()
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(HARD_TRAIN):.4f}")

tb2.eval()
print("\nEval with retrained last-token thinking block:")
for scale in [0.1, 0.05, 0.01]:
    correct = 0
    for item in EVAL:
        inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
            phi_x = embeds[:, -1, :]
            h = phi_x.clone()
            for _ in range(3):
                h = tb2(h, phi_x)
            delta = (h - phi_x).unsqueeze(1) * scale
            modified = embeds.clone().to(model.dtype)
            modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
            gen = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
                max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        nums = re.findall(r"\d+(?:\.\d+)?", decoded)
        if item["answer"] in nums: correct += 1
    acc = correct/len(EVAL)*100
    print(f"Scale {scale}: {acc:.1f}% | vs baseline: {acc-66.7:+.1f}%")


Retraining with last-token phi_x...
Epoch 20 | Loss: 0.0002
Epoch 40 | Loss: 0.0000
Epoch 60 | Loss: 0.0000
Epoch 80 | Loss: 0.0000
Epoch 100 | Loss: 0.0000

Eval with retrained last-token thinking block:
Scale 0.1: 73.3% | vs baseline: +6.6%
Scale 0.05: 60.0% | vs baseline: -6.7%
Scale 0.01: 53.3% | vs baseline: -13.4%


In [ ]:
# Full detailed results at the winning configuration
print("FINAL RESULTS — Last Token phi_x, Scale 0.1, 100 epochs")
print("="*60)
print(f"{'Question':>40} | Base | LCLDD")
print("-"*60)

baseline_correct = 0
lcldd_correct = 0

for item in EVAL:
    inputs = tok(item["question"], return_tensors="pt", truncation=True, max_length=64).to(DEVICE)
    with torch.no_grad():
        # Baseline
        gen_b = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        dec_b = tok.decode(gen_b[0], skip_special_tokens=True)
        hit_b = item["answer"] in re.findall(r"\d+(?:\.\d+)?", dec_b)

        # LCLDD last-token
        embeds = model.get_input_embeddings()(inputs["input_ids"]).float()
        phi_x = embeds[:, -1, :]
        h = phi_x.clone()
        for _ in range(3):
            h = tb2(h, phi_x)
        delta = (h - phi_x).unsqueeze(1) * 0.1
        modified = embeds.clone().to(model.dtype)
        modified[:, -1:, :] = modified[:, -1:, :] + delta.to(model.dtype)
        gen_l = model.generate(inputs_embeds=modified, attention_mask=inputs["attention_mask"],
            max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
        dec_l = tok.decode(gen_l[0], skip_special_tokens=True)
        hit_l = item["answer"] in re.findall(r"\d+(?:\.\d+)?", dec_l)

    if hit_b: baseline_correct += 1
    if hit_l: lcldd_correct += 1

    status = ""
    if not hit_b and hit_l: status = "<-- FIXED by LCLDD ✅"
    if hit_b and not hit_l: status = "<-- broke"

    ans_short = item["question"][:35]
    print(f"Exp:{item['answer']:>6} | {'V' if hit_b else 'X'}    | {'V' if hit_l else 'X'}  {status}")

print("="*60)
print(f"Baseline:    {baseline_correct}/{len(EVAL)} = {baseline_correct/len(EVAL)*100:.1f}%")
print(f"LCLDD:       {lcldd_correct}/{len(EVAL)} = {lcldd_correct/len(EVAL)*100:.1f}%")
print(f"Improvement: +{(lcldd_correct-baseline_correct)/len(EVAL)*100:.1f}% ✅")
print("="*60)

# Save winning checkpoint
os.makedirs("checkpoints", exist_ok=True)
torch.save({
    "thinking_block_state": tb2.state_dict(),
    "config": {
        "phi_x": "last_token",
        "scale": 0.1,
        "hidden_dim": model.config.hidden_size,
        "T_max": 3,
        "epochs": 100
    },
    "final_losses": {"L_lya": 0.0, "L_vf": 0.0},
    "results": {
        "baseline": baseline_correct/len(EVAL),
        "lcldd": lcldd_correct/len(EVAL),
        "improvement": (lcldd_correct-baseline_correct)/len(EVAL)
    }
}, "checkpoints/lcldd_winning.pt")
print("Winning checkpoint saved!")

# Save to Drive
import shutil
os.makedirs("/content/drive/MyDrive/lcldd_winning", exist_ok=True)
shutil.copy("checkpoints/lcldd_winning.pt",
            "/content/drive/MyDrive/lcldd_winning/lcldd_winning.pt")
print("Saved to Drive!")

FINAL RESULTS — Last Token phi_x, Scale 0.1, 100 epochs
                                Question | Base | LCLDD
------------------------------------------------------------
Exp:     4 | V    | V  
Exp:    15 | V    | V  
Exp:     3 | V    | V  
Exp:    42 | V    | V  
Exp:    72 | V    | V  
Exp:    40 | V    | V  
Exp:    32 | X    | X  
Exp:    48 | V    | X  <-- broke
Exp:   130 | X    | V  <-- FIXED by LCLDD ✅
Exp:   200 | X    | X  
Exp:   135 | V    | V  
Exp:    10 | V    | V  
Exp:    18 | X    | X  
Exp:    30 | X    | V  <-- FIXED by LCLDD ✅
Exp:    60 | V    | V  
Baseline:    10/15 = 66.7%
LCLDD:       11/15 = 73.3%
Improvement: +6.7% ✅
Winning checkpoint saved!
Saved to Drive!
